## EDA Paso 1: Exploración Inicial

In [0]:
-- ============================================================
-- EDA 1.1: ¿Cuántos registros tenemos?
-- ============================================================

SELECT COUNT(*) as total_registros
FROM analisis_tmdb.bronze.genres_bronze

In [0]:
-- ============================================================
-- EDA 1.2: Ver las primeras filas
-- ============================================================

SELECT *
FROM analisis_tmdb.bronze.genres_bronze
LIMIT 10;

## EDA Paso 2: Análisis de Valores Nulos

In [0]:
-- ============================================================
-- EDA 2.1: Contar valores nulos por columna
-- ============================================================

SELECT
    SUM(CASE WHEN genre_id IS NULL THEN 1 ELSE 0 END) AS null_id,
    SUM(CASE WHEN genre_name IS NULL THEN 1 ELSE 0 END) AS null_name
FROM analisis_tmdb.bronze.genres_bronze;

In [0]:
-- ============================================================
-- EDA 3.1: Distribución de genre_name
-- ============================================================

SELECT 
    genre_name,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM analisis_tmdb.bronze.genres_bronze
GROUP BY genre_name
ORDER BY cantidad DESC;

## EDA Paso 4: Estadísticas Descriptivas de Variables Numéricas

Este dataset no tiene variables numericas con las cuales realizar calculos


## EDA Paso 5: Detección de Problemas de Calidad

In [0]:
-- ============================================================
-- EDA 5.1: Reporte de calidad de datos usando CTEs
-- ============================================================

WITH total AS (
    SELECT COUNT(*) as n 
    FROM analisis_tmdb.bronze.genres_bronze
),
problemas AS (
    SELECT
        -- ID vacío
        COUNT(CASE WHEN genre_id IS NULL THEN 1 END) as id_vacio,
        -- Genre vacío
        COUNT(CASE WHEN genre_name IS NULL OR genre_name = '' THEN 1 END) as genre_vacio
    FROM analisis_tmdb.bronze.genres_bronze
)
SELECT 
    t.n as total_registros,
    p.id_vacio,
    ROUND(p.id_vacio * 100.0 / t.n, 2) as pct_id_vacio,
    p.genre_vacio,
    ROUND(p.genre_vacio * 100.0 / t.n, 2) as pct_genre_vacio
FROM total t, problemas p;

In [0]:
-- ============================================================
-- EDA 5.2: Detectar posibles duplicados
-- ============================================================

SELECT
    genre_id,
    COUNT(*) AS cantidad
FROM analisis_tmdb.bronze.genres_bronze
GROUP BY genre_id
HAVING COUNT(*) > 1;

In [0]:
CREATE OR REPLACE TEMP VIEW bronze_EDA_genre AS (
SELECT
    genre_id,
    -- Validar código de idioma (2 letras)
    CASE
        WHEN genre_name RLIKE '^[a-zA-Z0-9 :''.,!?-]+$'
        THEN genre_name
        ELSE NULL
    END AS genre_name
FROM analisis_tmdb.bronze.genres_bronze);

In [0]:
select * from bronze_EDA_genre limit 10

# =============================
# CONCLUSIONES 
# =============================

## Hallazgos del EDA

### Problemas de calidad identificados:

Los valores que se obtienen de la API vienen limpios ya que es un dataset simple